# Эффективность рекламных кампаний

Чистый Python, без pandas. Данные: `data/campaign_data.csv`, 1020 кампаний, платформы Telegram, Вконтакте, Instagram, Google, Яндекс, 8 городов, период 2022 год.

Постановка в `data/03_рекламные_кампании.ТЗ.md`.

## Данные

```
ID Кампании, Начальная дата, Конечная дата, Тип кампании,
Платформа, Бюджет, Город, Показы, Клики, Конверсия, Доход
```

Числовые поля приходят строками, часть значений с плавающей точкой, например `507.0` в кликах. Поэтому все числа переводятся через `float` с последующим `int`, прямой `int("507.0")` падает.

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta

DATA = "../data/campaign_data.csv"
HEADER = ["id", "start", "end", "type", "platform", "budget",
          "city", "shows", "clicks", "conversions", "revenue"]
COL = {name: i for i, name in enumerate(HEADER)}

with open(DATA, encoding="utf-8") as f:
    f.readline()                       # заголовок
    rows = [line.rstrip("\n").split(",") for line in f]

num = lambda row, key: int(float(row[COL[key]]))

print("кампаний:", len(rows))
print("полей в строке:", len(rows[0]))
print("пример:", rows[0][:7])
print("клики в данных:", rows[0][COL["clicks"]], "-> int(float(...)) =", num(rows[0], "clicks"))

## Задача 1. Какая платформа эффективнее

В ТЗ эффективность определена как среднее число конверсий на кампанию. Считаю именно так, но результат покажу в двух метриках, потому что выбор меняет ответ.

**Среднее число конверсий** отвечает на вопрос «сколько конверсий приносит одна кампания». **Доход на рубль бюджета** отвечает на вопрос «выгодно ли тратить деньги сюда». Это разные вещи, и маркетологу нужна вторая.

In [ ]:
def avg_conversions(rows, key):
    total = defaultdict(int)
    count = defaultdict(int)
    for r in rows:
        total[r[COL[key]]] += num(r, "conversions")
        count[r[COL[key]]] += 1
    return {k: round(total[k] / count[k], 2) for k in total}

def roi(rows, key):
    revenue = defaultdict(int)
    budget = defaultdict(int)
    count = defaultdict(int)
    for r in rows:
        revenue[r[COL[key]]] += num(r, "revenue")
        budget[r[COL[key]]] += num(r, "budget")
        count[r[COL[key]]] += 1
    return {k: round(revenue[k] / budget[k], 3) for k in revenue}, count

avg_plat = avg_conversions(rows, "platform")
roi_plat, cnt_plat = roi(rows, "platform")

print("Платформа     ср.конверсий   доход/бюджет   кампаний")
for k in sorted(roi_plat, key=lambda x: -roi_plat[x]):
    print(f"{k:<14} {avg_plat[k]:>8.2f}      {roi_plat[k]:>6.3f}      {cnt_plat[k]:>4}")

По среднему числу конверсий лидирует Instagram (107.8), по доходу на рубль бюджета тоже Instagram (0.677). А вот по конверсиям на клик лидирует Telegram, и это другой ответ.

Разница видна на примере Яндекса: 95.3 конверсии на кампанию, худший показатель, но по конверсиям на клик он 0.176, и по доходу 0.544. Кампании на Яндексе просто дорогие, поэтому на единицу денег они дают меньше всех.

Среднее число конверсий само по себе вводит в заблуждение: оно не учитывает ни стоимость, ни качество трафика. Если по условию задачи надо назвать самую эффективную платформу, честный ответ для бизнеса это «по деньгам», и его стоит проговорить отдельно от формального ответа по ТЗ.

In [ ]:
print("Платформа     конверсий/клик   доход/бюджет")
for k in sorted(roi_plat, key=lambda x: -roi_plat[x]):
    conv = sum(num(r, "conversions") for r in rows if r[COL["platform"]] == k)
    clicks = sum(num(r, "clicks") for r in rows if r[COL["platform"]] == k)
    print(f"{k:<14}   {conv/clicks:.4f}        {roi_plat[k]:.3f}")

## Задача 2. Тип кампании и город

In [ ]:
for key, title in [("type", "Тип кампании"), ("city", "Город")]:
    avg = avg_conversions(rows, key)
    roi_k, cnt = roi(rows, key)
    print(f"=== {title}")
    print(f"{'Значение':<28} ср.конверсий  доход/бюджет  кампаний")
    for k in sorted(roi_k, key=lambda x: -roi_k[x]):
        print(f"{k:<28} {avg[k]:>8.2f}     {roi_k[k]:>6.3f}     {cnt[k]:>4}")
    print()

Разброс по городам шире, чем по платформам: доход на бюджет от 0.512 в Калуге до 0.694 в Казани, разница 36%. По платформам тот же размах 0.544-0.677, около 24%.

Казань получает 12.15% бюджета, это примерно средний уровень по городам, и при этом отдаёт лучше всех. Москва получает больше, 11.63%, но отдаёт хуже. Разница между лучшим и худшим городом по эффективности 36%, и она не объясняется разницей в объёме вложений.

In [ ]:
city_budget = defaultdict(int)
city_roi = {}
for r in rows:
    city_budget[r[COL["city"]]] += num(r, "budget")
roi_city, _ = roi(rows, "city")

total_budget = sum(city_budget.values())
print(f"{'Город':<20} {'бюджет':>10} {'доля':>7} {'доход/бюджет':>13}")
for c_ in sorted(roi_city, key=lambda x: -roi_city[x]):
    print(f"{c_:<20} {city_budget[c_]:>10} {100*city_budget[c_]/total_budget:>6.2f}% {roi_city[c_]:>13.3f}")

## Задача 3. Доли бюджета по городам внутри платформы

Здесь важно не запутать два разреза. Доля города в бюджете своей платформы и доля города в бюджете всего проекта это разные числа: во втором случае города сравниваются между платформами, и город с большой долей на одной платформе потеряется.

In [ ]:
def budget_share(rows, platform):
    by_city = defaultdict(int)
    total = 0
    for r in rows:
        if r[COL["platform"]] != platform:
            continue
        amount = num(r, "budget")
        by_city[r[COL["city"]]] += amount
        total += amount
    return {c: round(100 * v / total, 2) for c, v in sorted(by_city.items(), key=lambda x: -x[1])}

shares = budget_share(rows, "Instagram")
print("Instagram, доля города в бюджете платформы:")
for c_, v in shares.items():
    print(f"  {c_:<20} {v:>6.2f}%  {'#' * int(v)}")

На Instagram Москва получает 9.11% бюджета, последнее место из восьми городов, при том что по доходу на бюджет Москва в среднем по платформам 0.628, это четвёртое место из восьми. Деньги в Москву не вкладываются почти так же, как в Калугу с результатом 0.543.

Одновременно Казань на Instagram получает 12.22% при лучшем результате. Перекос есть в обе стороны, но в сторону Казани он работает.

## Задача 4. Даты без кампаний

Генератор пропущенных дат: строится полный ряд между минимальной и максимальной датой запуска, из него вычитаются дни, в которые хоть что-то запускалось.

In [ ]:
def missing_dates(rows):
    started = {r[COL["start"]] for r in rows}
    first = datetime.strptime(min(started), "%Y-%m-%d")
    last = datetime.strptime(max(started), "%Y-%m-%d")
    days, cursor = [], first
    while cursor <= last:
        key = cursor.strftime("%Y-%m-%d")
        if key not in started:
            days.append(key)
        cursor += timedelta(days=1)
    return days

gaps = missing_dates(rows)
print(f"период: {min(r[COL['start']] for r in rows)} — {max(r[COL['start']] for r in rows)}")
print(f"дней без запусков: {len(gaps)}")
print(gaps)

18 дней без запусков за год, идут они кластерами: подряд 24 и 25 июня, затем 29 и 30 июня, дальше по одному 31 июля, 10 и 29 августа.

Пустой день не всегда значит проблему. Июль и август это сезонный провал в рекламной активности, а 1-13 января нормальное затишье после праздников. Но равномерный запуск был бы более предсказуемым.

## Задача 5. Кампании по городу с бюджетом выше порога

In [ ]:
def campaigns_above(rows, city, budget_threshold):
    result = []
    for r in rows:
        if r[COL["city"]] == city and num(r, "budget") > budget_threshold:
            result.append({
                "ID Кампании": r[COL["id"]],
                "Тип": r[COL["type"]],
                "Платформа": r[COL["platform"]],
                "Бюджет": num(r, "budget"),
                "Доход": num(r, "revenue"),
            })
    return sorted(result, key=lambda x: int(x["ID Кампании"]))

big = campaigns_above(rows, "Казань", 40000)
print(f"Казань, бюджет выше 40 000: {len(big)} кампаний")
for c_ in big[:8]:
    print(f"  {c_['ID Кампании']:>5}  {c_['Платформа']:<12} {c_['Бюджет']:>7}  доход {c_['Доход']:>7}  ROI {c_['Доход']/c_['Бюджет']:.3f}")

## Итоги

| Метрика | Значение |
|---|---|
| Кампаний в данных | 1020 |
| Лучшая платформа по ср. конверсий | Instagram, 107.83 |
| Лучшая платформа по доходу на бюджет | Instagram, 0.677 |
| Лучшая платформа по конверсиям на клик | Telegram, 0.1966 |
| Худшая платформа по доходу на бюджет | Яндекс, 0.544 |
| Лучший город по доходу на бюджет | Казань, 0.694 |
| Худший город по доходу на бюджет | Калуга, 0.512 |
| Дней без запусков за год | 18 |

Что стоит вынести:

1. Среднее число конверсий на кампанию и доход на рубль бюджета дают разные ответы. Яндекс худший по первому и предпоследний по второму, потому что кампании дорогие. Для решения о бюджете нужен второй показатель, первый отвечает на вопрос про формат кампаний.
2. Разброс между городами шире, чем между платформами: 36% против 24%. География оказалась более значимым признаком, чем выбор площадки.
3. Москва на Instagram получает 9.11% бюджета, последнее место из восьми, при среднем результате. Перекос работает в сторону Казани, который получает 12.22% и отдаёт лучше.
4. Все ROI в данных меньше единицы: доход ниже бюджета на любой платформе и в любом городе. Либо в столбце «доход» не выручка, либо реклама действительно убыточна. Этот вопрос стоит задать до того, как делать выводы о перераспределении.